# 🔬 TranSTR Inference & Attention Analysis

**Features:**
1. Download best checkpoint from W&B
2. Full test-set evaluation → D, E, PA, PR, PAR, CA, CR, CAR, ALL
3. Save incorrect predictions to CSV
4. Single-sample inference with attention extraction
5. Rich attention visualizations explaining model reasoning

In [ ]:
# CELL 1: Git Clone & Setup
import os
REPO_URL = "https://github.com/DanielQH07/tranSTR_Casual.git"
REPO_NAME = "tranSTR_Casual"
BRANCH = "groundedDINO"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} -b {BRANCH}
else:
    print("Repo already exists.")

if os.path.basename(os.getcwd()) != REPO_NAME:
    try:
        target_dir = os.path.join(os.getcwd(), REPO_NAME, "causalvid")
        if os.path.exists(target_dir):
            os.chdir(target_dir)
        elif os.path.exists(REPO_NAME):
            os.chdir(REPO_NAME)
        print(f"Changed directory to: {os.getcwd()}")
    except Exception as e:
        print(f"Could not set working directory: {e}")

In [ ]:
# CELL 2: Install & W&B Login
print('=== Installing dependencies ===')
!pip install -q wandb peft --upgrade
import wandb

# ============================================
# 🔴 PASTE YOUR W&B API KEY HERE
# ============================================
WANDB_API_KEY = ''
WANDB_PROJECT = 'transtr-causalvid-dino-lora'
WANDB_ENTITY = None

wandb.login(key=WANDB_API_KEY)
print('✅ W&B logged in!')

In [ ]:
# CELL 3: Imports
print('=== Imports ===')
import os, copy, math, torch, numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast
from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel
from tqdm.auto import tqdm
from itertools import chain
from einops import rearrange

# Set style for all plots
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'figure.figsize': (14, 8),
    'font.size': 11,
})
print('Imports OK')

In [ ]:
# CELL 4: Configuration
print('=== Configuration ===')

# ============================================
# 🔴 UPDATE THESE PATHS FOR YOUR KAGGLE SETUP
# ============================================
CLIP_SPLIT_PATH = '/kaggle/input/datasets/danielq07/dinov3-feat/features'
CLIP_MERGED_PATH = '/kaggle/working/dinov3_T16_dim1024_merge'
GDINO_FEATURE_PATH = '/kaggle/input/datasets/danielq07/gdino-roi-all-nodes-merged'
ANNOTATION_PATH = '/kaggle/input/text-annotation/QA'
SPLIT_DIR = '/kaggle/input/casual-vid-data-split/split'
BASE = '/kaggle/working'
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

FEAT_DIM = 1024
BEST_ARTIFACT_NAME = 'best-model-dinov3'

class Config:
    video_feature_root = CLIP_MERGED_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR
    topK_frame = 16
    objs = 20
    frames = 16
    select_frames = 5
    topK_obj = 12
    frame_feat_dim = FEAT_DIM
    obj_feat_dim = 1028
    use_grounding_dino = True
    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3
    text_encoder_type = 'microsoft/deberta-v3-base'
    freeze_text_encoder = False
    text_encoder_lr = 1e-5
    text_pool_mode = 1
    use_lora = True
    lora_r = 16
    lora_alpha = 32
    lora_dropout = 0.15
    lora_target_modules = ['query_proj', 'key_proj', 'value_proj']
    lora_lr = 5e-5
    bs = 8
    n_query = 5
    return_family_id = True
    hard_eval = False
    num_workers = 4

args = Config()
set_gpu_devices(0)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

QTYPE_DISPLAY = {
    'descriptive': 'D  (Descriptive)',
    'explanatory': 'E  (Explanatory)',
    'predictive': 'PA (Predictive Answer)',
    'predictive_reason': 'PR (Predictive Reason)',
    'counterfactual': 'CA (Counterfactual Answer)',
    'counterfactual_reason': 'CR (Counterfactual Reason)',
}

print(f'Device: {device}')
print(f'Config: frame_feat_dim={args.frame_feat_dim}, obj_feat_dim={args.obj_feat_dim}')

In [ ]:
# CELL 5: Merge Features
import shutil
from tqdm.auto import tqdm

print(f'\nSource: {CLIP_SPLIT_PATH}')
if os.path.exists(CLIP_SPLIT_PATH):
    subfolders = os.listdir(CLIP_SPLIT_PATH)
    print(f'Subfolders found: {subfolders}')
else:
    print('❌ Source path not found!')
    subfolders = []

os.makedirs(CLIP_MERGED_PATH, exist_ok=True)
for split in ['train', 'test', 'valid', 'val']:
    split_folder = os.path.join(CLIP_SPLIT_PATH, split)
    if not os.path.exists(split_folder):
        continue
    pt_files = [f for f in os.listdir(split_folder) if f.endswith('.pt')]
    print(f'\n📁 {split}: {len(pt_files)} files')
    for fname in tqdm(pt_files, desc=f'Copying {split}'):
        src = os.path.join(split_folder, fname)
        dst = os.path.join(CLIP_MERGED_PATH, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)

final_count = len([f for f in os.listdir(CLIP_MERGED_PATH) if f.endswith('.pt')])
print(f'\n✅ Merge complete! Total .pt files: {final_count}')

In [ ]:
# CELL 6: Download Best Checkpoint from W&B
print('=== Downloading Best Checkpoint ===')
import wandb

api = wandb.Api()
entity = WANDB_ENTITY or api.default_entity
artifact_path = f'{entity}/{WANDB_PROJECT}/{BEST_ARTIFACT_NAME}:best'
print(f'📥 Downloading: {artifact_path}')

artifact = api.artifact(artifact_path)
artifact_dir = artifact.download()
checkpoint_path = os.path.join(artifact_dir, 'best_model_state_dict.ckpt')

print(f'✅ Downloaded to: {checkpoint_path}')
if artifact.metadata:
    print(f'📊 Checkpoint metadata:')
    for k, v in artifact.metadata.items():
        print(f'   {k}: {v}')

In [ ]:
# CELL 7: Build Model & Load Checkpoint
print('=== Building Model ===')

cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames

model = VideoQAmodel(**cfg)
model.to(device)

state_dict = torch.load(checkpoint_path, map_location=device)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f'✅ Model loaded!')
print(f'   Missing keys:    {len(missing)}')
print(f'   Unexpected keys: {len(unexpected)}')
if missing:
    print(f'   (Missing: {missing[:5]}...)')

model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'   Total params: {total_params/1e6:.1f}M')

In [ ]:
# CELL 8: Create Test Dataset
print('=== Creating Test Dataset ===')

test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True,
    return_family_id=args.return_family_id
)

test_loader = DataLoader(test_ds, args.bs, shuffle=False,
                         num_workers=args.num_workers, pin_memory=True)

print(f'\n✅ Test: {len(test_ds)} samples -> {len(test_loader)} batches')

In [ ]:
# CELL 9: Full Evaluation - All Metrics
print('=== Full Evaluation ===')

def evaluate_all_metrics(model, loader, device):
    # Evaluate and return per-type results + all metrics.
    model.eval()
    type_results = {}
    all_predictions = []
    
    # Build meta map from dataset
    ds = loader.dataset
    meta_map = {}
    for _, row in ds.sample_list.iterrows():
        vid = str(row.get('video_id', ''))
        qtype = str(row.get('type', 'unknown'))
        qns_key = f'{vid}_{qtype}'
        meta_map[qns_key] = {
            'video_id': vid,
            'question_type': qtype,
            'question': str(row.get('question', '')),
            'answer_idx': int(row.get('answer', -1)),
            'answers': [str(row.get(f'a{i}', '')) for i in range(5)]
        }
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            if len(batch) == 7:
                ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = batch
            else:
                ff, of, qns, ans_word, ans_id, qns_keys = batch
                q_family_id = None
            
            ff, of = ff.to(device), of.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            
            out = model(ff, of, qns, ans_word, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out.get('logits', out))
            if isinstance(logits, dict):
                logits = logits['logits']
            
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds = probs.argmax(axis=-1)
            targets = ans_id.numpy()
            
            for si, (key, pred, target, pv) in enumerate(zip(qns_keys, preds, targets, probs)):
                meta = meta_map.get(str(key), {})
                vid = meta.get('video_id', str(key).rsplit('_', 1)[0])
                qtype = meta.get('question_type', 'unknown')
                question = meta.get('question', '')
                answers = meta.get('answers', [''] * 5)
                
                is_correct = int(pred) == int(target)
                
                if qtype not in type_results:
                    type_results[qtype] = []
                type_results[qtype].append({
                    'video_id': vid, 'pred': int(pred),
                    'target': int(target), 'correct': is_correct
                })
                
                all_predictions.append({
                    'video_id': vid, 'question_type': qtype,
                    'question': question,
                    'correct_idx': int(target), 'predicted_idx': int(pred),
                    'is_correct': int(is_correct),
                    'confidence': float(pv[int(pred)]),
                    'correct_answer': answers[int(target)] if int(target) < len(answers) else '',
                    'predicted_answer': answers[int(pred)] if int(pred) < len(answers) else '',
                    **{f'a{i}': answers[i] for i in range(5)},
                    **{f'prob_a{i}': float(pv[i]) for i in range(5)},
                })
    
    # Compute metrics
    metrics = {}
    qtypes_single = ['descriptive', 'explanatory', 'predictive',
                     'predictive_reason', 'counterfactual', 'counterfactual_reason']
    short_names = {'descriptive': 'D', 'explanatory': 'E',
                   'predictive': 'PA', 'predictive_reason': 'PR',
                   'counterfactual': 'CA', 'counterfactual_reason': 'CR'}
    
    for qt in qtypes_single:
        results = type_results.get(qt, [])
        correct = sum(1 for r in results if r['correct'])
        total = len(results)
        acc = correct / total * 100 if total > 0 else 0
        metrics[short_names[qt]] = {'acc': acc, 'correct': correct, 'total': total}
    
    # PAR: predictive answer + reason both correct
    def calc_paired(ans_type, reason_type, name):
        ans_by_vid = {r['video_id']: r['correct'] for r in type_results.get(ans_type, [])}
        reason_by_vid = {r['video_id']: r['correct'] for r in type_results.get(reason_type, [])}
        common = set(ans_by_vid.keys()) & set(reason_by_vid.keys())
        both = sum(1 for v in common if ans_by_vid[v] and reason_by_vid[v])
        total = len(common)
        acc = both / total * 100 if total > 0 else 0
        metrics[name] = {'acc': acc, 'correct': both, 'total': total}
    
    calc_paired('predictive', 'predictive_reason', 'PAR')
    calc_paired('counterfactual', 'counterfactual_reason', 'CAR')
    
    # ALL = (D + E + PAR + CAR) / 4
    all_acc = (metrics['D']['acc'] + metrics['E']['acc'] +
               metrics['PAR']['acc'] + metrics['CAR']['acc']) / 4
    metrics['ALL'] = {'acc': all_acc, 'correct': '-', 'total': '-'}
    
    return metrics, all_predictions

metrics, all_predictions = evaluate_all_metrics(model, test_loader, device)

# Print formatted results
print('\n' + '═' * 65)
print('  📊 CAUSALVIDQA EVALUATION RESULTS (TEST SET)')
print('═' * 65)
print(f'  {"Metric":<30} {"Correct":>8} {"Total":>8} {"Accuracy":>10}')
print('─' * 65)
for name in ['D', 'E', 'PA', 'PR', 'CA', 'CR']:
    m = metrics[name]
    full_name = {'D': 'Descriptive', 'E': 'Explanatory',
                 'PA': 'Predictive Answer', 'PR': 'Predictive Reason',
                 'CA': 'Counterfactual Answer', 'CR': 'Counterfactual Reason'}[name]
    print(f'  {name:<4} ({full_name:<23}) {m["correct"]:>8} {m["total"]:>8} {m["acc"]:>9.2f}%')
print('─' * 65)
for name in ['PAR', 'CAR']:
    m = metrics[name]
    full = 'Predictive A+R' if name == 'PAR' else 'Counterfactual A+R'
    print(f'  {name:<4} ({full:<23}) {m["correct"]:>8} {m["total"]:>8} {m["acc"]:>9.2f}%')
print('─' * 65)
m = metrics['ALL']
print(f'  {"ALL":<4} {"(D+E+PAR+CAR)/4":<24} {"":>8} {"":>8} {m["acc"]:>9.2f}%')
print('═' * 65)

# Bar chart
fig, ax = plt.subplots(figsize=(14, 6))
keys = ['D', 'E', 'PA', 'PR', 'PAR', 'CA', 'CR', 'CAR', 'ALL']
vals = [metrics[k]['acc'] for k in keys]
colors = ['#58a6ff', '#58a6ff', '#3fb950', '#3fb950', '#1f6feb',
          '#f0883e', '#f0883e', '#da3633', '#a371f7']
bars = ax.bar(keys, vals, color=colors, edgecolor='#30363d', linewidth=1.2)
ax.set_ylim(0, 100)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('CausalVidQA Evaluation Results', fontsize=16, fontweight='bold', color='#f0f6fc')
ax.grid(axis='y', linestyle='--', alpha=0.3)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1.5,
            f'{v:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10, color='#f0f6fc')
plt.tight_layout()
plt.savefig('eval_results_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to eval_results_all_metrics.png')

In [ ]:
# CELL 10: Save Incorrect Predictions to CSV
print('=== Saving Incorrect Predictions ===')

pred_df = pd.DataFrame(all_predictions)
incorrect_df = pred_df[pred_df['is_correct'] == 0].copy()
incorrect_df = incorrect_df.sort_values(['question_type', 'confidence'], ascending=[True, False])

# Save all predictions
pred_df.to_csv('all_predictions.csv', index=False)
print(f'📝 All predictions saved: all_predictions.csv ({len(pred_df)} rows)')

# Save incorrect only
incorrect_df.to_csv('incorrect_predictions.csv', index=False)
print(f'❌ Incorrect predictions saved: incorrect_predictions.csv ({len(incorrect_df)} rows)')

# Summary by type
print(f'\n📊 Incorrect by question type:')
for qt in incorrect_df['question_type'].unique():
    n = len(incorrect_df[incorrect_df['question_type'] == qt])
    total = len(pred_df[pred_df['question_type'] == qt])
    print(f'   {qt:<25}: {n}/{total} incorrect ({n/total*100:.1f}%)')

# Show a few examples
print(f'\n📋 Sample incorrect predictions:')
sample = incorrect_df.head(5)
for _, row in sample.iterrows():
    print(f'\n  Video: {row["video_id"]}')
    print(f'  Type:  {row["question_type"]}')
    print(f'  Q:     {row["question"][:80]}...')
    print(f'  ✓ Correct: [{row["correct_idx"]}] {row["correct_answer"][:50]}')
    print(f'  ✗ Predicted: [{row["predicted_idx"]}] {row["predicted_answer"][:50]} (conf={row["confidence"]:.3f})')

In [ ]:
# CELL 11: Inference with Attention Extraction
print('=== Defining inference_with_attention ===')

from networks.topk import PerturbedTopK, HardtopK

@torch.no_grad()
def inference_with_attention(model, frame_feat, obj_feat, qns_word, ans_word, device, q_family_id=None):
    # Run single-sample inference through the model step-by-step,
    # capturing attention weights at every stage.
    # Returns dict with predictions, probabilities, and all attention maps.
    model.eval()
    B, F_total, O = obj_feat.size()[:3]
    
    # ── Step 1: Resize frame features ──
    frame_feat_proj = model.frame_resize(frame_feat)  # [B, F, d_model]
    
    # ── Step 2: Encode question ──
    q_local, q_mask = model.forward_text(list(qns_word), device)  # [B, q_len, d_model]
    q_len = q_local.size(1)
    
    # ── Step 3: Frame decoder → question-aware frame selection ──
    frame_mask = torch.ones(B, F_total).bool().to(device)
    frame_local, frame_att = model.frame_decoder(
        frame_feat_proj, q_local,
        memory_key_padding_mask=q_mask,
        query_pos=model.pos_encoder_1d(frame_mask, model.d_model),
        output_attentions=True
    )
    # frame_att: [B, F, q_len] - frame-to-question cross-attention
    frame_importance = frame_att.mean(dim=-1)  # [B, F] average over q tokens
    
    # ── Step 4: TopK frame selection ──
    idx_frame = rearrange(
        model.frame_sorter(frame_att.flatten(1, 2)),
        'b (f q) k -> b f q k', f=F_total
    ).sum(-2)  # [B, F, topK]
    
    # Find which frames were selected
    selected_indices = idx_frame[0].argmax(dim=0).cpu().numpy()  # [topK]
    frame_selection_weights = idx_frame[0].cpu().numpy()  # [F, topK]
    
    frame_local = (frame_local.transpose(1, 2) @ idx_frame).transpose(1, 2)
    
    # ── Step 5: Object processing ──
    obj_feat_sel = (obj_feat.flatten(-2, -1).transpose(1, 2) @ idx_frame).transpose(1, 2)
    obj_feat_sel = obj_feat_sel.view(B, model.frame_topK, O, -1)
    obj_local = model.obj_resize(obj_feat_sel)
    
    q_rep = q_local.repeat_interleave(model.frame_topK, dim=0)
    q_mask_rep = q_mask.repeat_interleave(model.frame_topK, dim=0)
    
    obj_local, obj_att = model.obj_decoder(
        obj_local.flatten(0, 1), q_rep,
        memory_key_padding_mask=q_mask_rep,
        output_attentions=True
    )
    # obj_att: [B*topK, O, q_len]
    obj_importance = obj_att.mean(dim=-1).view(B, model.frame_topK, O)  # [B, topK, O]
    
    if model.use_grounding_dino:
        obj_local = obj_local.view(B, model.frame_topK, O, -1)
    else:
        idx_obj = rearrange(
            model.obj_sorter(obj_att.flatten(1, 2)),
            'b (o q) k -> b o q k', o=O
        ).sum(-2)
        obj_local = (obj_local.transpose(1, 2) @ idx_obj).transpose(1, 2)
        obj_local = obj_local.view(B, model.frame_topK, model.obj_topK, -1)
    
    # ── Step 6: Frame-Object hierarchy ──
    frame_obj, fo_att = model.fo_decoder(
        frame_local, obj_local.flatten(1, 2),
        output_attentions=True
    )
    
    # ── Step 7: Vision-Language fusion ──
    frame_mask_topk = torch.ones(B, model.frame_topK).bool().to(device)
    frame_obj = frame_obj.view(B, model.frame_topK, -1)
    frame_qns_mask = torch.cat((frame_mask_topk, q_mask), dim=1).bool()
    
    mem = model.vl_encoder(
        torch.cat((frame_obj, q_local), dim=1),
        src_key_padding_mask=frame_qns_mask,
        pos=model.pos_encoder_1d(frame_qns_mask.bool(), model.d_model)
    )
    
    # ── Step 8: Encode answers ──
    a_seq, _ = model.forward_text(list(chain(*ans_word)), device, has_ans=True)
    a_seq = rearrange(a_seq, '(n b) t c -> b n t c', b=B)
    tgt = a_seq[:, :, 0, :]  # [CLS] tokens: [B, 5, d_model]
    
    # ── Step 9: Answer decoder ──
    out, ans_att = model.ans_decoder(
        tgt, mem,
        memory_key_padding_mask=frame_qns_mask,
        output_attentions=True
    )
    # ans_att: [B, 5, mem_len] where mem_len = topK + q_len
    
    # ── Step 10: Scoring ──
    cand_feat, answer_score, evidence_score = model.decode_candidates(out)
    mem_pool = model.pool_memory(mem, mem_mask=frame_qns_mask)
    
    fused_score = answer_score
    knowledge_score = None
    if q_family_id is not None:
        if not isinstance(q_family_id, torch.Tensor):
            q_family_id = torch.tensor(q_family_id, dtype=torch.long, device=device)
        q_family_id = q_family_id.to(device).long().view(-1)
        k_feat = model._normalize_knowledge_feat(None, cand_feat)
        knowledge_score = model.score_knowledge_support(cand_feat, mem_pool, k_feat, q_family_id)
        fused_score = answer_score + model.lambda_knowledge * knowledge_score
    
    probs = F.softmax(fused_score, dim=-1)
    pred = probs.argmax(dim=-1)
    
    return {
        'pred': pred.cpu().item() if pred.numel() == 1 else pred.cpu().numpy(),
        'probs': probs[0].cpu().numpy(),
        'answer_score': answer_score[0].cpu().numpy(),
        'evidence_score': evidence_score[0].cpu().numpy(),
        'knowledge_score': knowledge_score[0].cpu().numpy() if knowledge_score is not None else None,
        'fused_score': fused_score[0].cpu().numpy(),
        'frame_importance': frame_importance[0].cpu().numpy(),       # [F]
        'frame_att': frame_att[0].cpu().numpy(),                     # [F, q_len]
        'selected_frame_indices': selected_indices,                  # [topK]
        'frame_selection_weights': frame_selection_weights,          # [F, topK]
        'obj_importance': obj_importance[0].cpu().numpy(),           # [topK, O]
        'ans_att': ans_att[0].cpu().numpy(),                         # [5, mem_len]
        'fo_att': fo_att[0].cpu().numpy() if fo_att is not None else None,
        'q_len': q_len,
        'topK': model.frame_topK,
    }

print('inference_with_attention defined')

In [ ]:
# CELL 12: Attention Visualization Functions
print('=== Defining Visualization Functions ===')

def visualize_full_attention(result, video_id, question, answers, correct_idx, question_type,
                             save_prefix='attention'):
    # Generate comprehensive attention visualizations.
    pred_idx = result['pred']
    probs = result['probs']
    frame_imp = result['frame_importance']
    sel_indices = result['selected_frame_indices']
    obj_imp = result['obj_importance']
    ans_att = result['ans_att']
    q_len = result['q_len']
    topK = result['topK']
    
    # ════════════════════════════════════════════════
    # FIGURE 1: Overview Dashboard
    # ════════════════════════════════════════════════
    fig = plt.figure(figsize=(20, 16))
    gs = gridspec.GridSpec(3, 2, hspace=0.4, wspace=0.3, height_ratios=[1, 1, 1.2])
    
    # Title
    status = '✅ CORRECT' if pred_idx == correct_idx else '❌ INCORRECT'
    fig.suptitle(
        f'{status}  |  Video: {video_id}  |  Type: {question_type}\n'
        f'Q: {question[:90]}{"..." if len(question) > 90 else ""}',
        fontsize=13, fontweight='bold', color='#f0f6fc', y=0.98
    )
    
    # ── Panel 1: Frame Importance ──
    ax1 = fig.add_subplot(gs[0, 0])
    n_frames = len(frame_imp)
    colors_frame = ['#1f6feb' if i in sel_indices else '#30363d' for i in range(n_frames)]
    bars1 = ax1.bar(range(n_frames), frame_imp, color=colors_frame, edgecolor='#484f58')
    ax1.set_xlabel('Frame Index')
    ax1.set_ylabel('Attention Score')
    ax1.set_title('Step 1: Frame Selection (question-aware)', fontsize=12, fontweight='bold', color='#58a6ff')
    ax1.set_xticks(range(n_frames))
    # Mark selected frames
    for idx in sel_indices:
        if idx < n_frames:
            ax1.annotate('★', (idx, frame_imp[idx]), ha='center', va='bottom',
                        fontsize=14, color='#ffd700')
    ax1.legend(['Selected ★', 'Not selected'], loc='upper right',
               facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')
    
    # ── Panel 2: Answer Probabilities ──
    ax2 = fig.add_subplot(gs[0, 1])
    ans_labels = [f'[{i}] {a[:35]}{"..." if len(a) > 35 else ""}' for i, a in enumerate(answers)]
    y_pos = range(len(answers))
    colors_ans = []
    for i in range(len(answers)):
        if i == correct_idx and i == pred_idx:
            colors_ans.append('#3fb950')  # green - correct prediction
        elif i == correct_idx:
            colors_ans.append('#58a6ff')  # blue - correct answer (missed)
        elif i == pred_idx:
            colors_ans.append('#da3633')  # red - wrong prediction
        else:
            colors_ans.append('#484f58')  # gray
    
    bars2 = ax2.barh(y_pos, probs, color=colors_ans, edgecolor='#30363d')
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(ans_labels, fontsize=9)
    ax2.set_xlabel('Probability')
    ax2.set_title('Final: Answer Probabilities', fontsize=12, fontweight='bold', color='#f0883e')
    ax2.set_xlim(0, 1)
    for i, (bar, p) in enumerate(zip(bars2, probs)):
        label = f'{p:.3f}'
        if i == correct_idx:
            label += ' ✓'
        if i == pred_idx:
            label += ' ◀ pred'
        ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                label, va='center', fontsize=9, color='#c9d1d9')
    ax2.invert_yaxis()
    
    # ── Panel 3: Object Attention per Selected Frame ──
    ax3 = fig.add_subplot(gs[1, :])
    obj_data = obj_imp[:, :min(10, obj_imp.shape[1])]  # Show top 10 objects
    im = ax3.imshow(obj_data, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax3.set_xlabel('Object Index (within frame)')
    ax3.set_ylabel('Selected Frame')
    ax3.set_yticks(range(topK))
    ax3.set_yticklabels([f'Frame {sel_indices[i]}' if i < len(sel_indices) else f'Slot {i}'
                         for i in range(topK)])
    ax3.set_title('Step 2: Object Attention in Selected Frames', fontsize=12,
                  fontweight='bold', color='#3fb950')
    plt.colorbar(im, ax=ax3, label='Attention', shrink=0.8)
    
    # ── Panel 4: Answer-to-Memory Attention ──
    ax4 = fig.add_subplot(gs[2, :])
    mem_len = ans_att.shape[1]
    im4 = ax4.imshow(ans_att, aspect='auto', cmap='magma', interpolation='nearest')
    ax4.set_xlabel('Memory Position')
    ax4.set_ylabel('Answer Candidate')
    ax4.set_yticks(range(5))
    ax4.set_yticklabels([f'[{i}] {a[:30]}...' if len(a) > 30 else f'[{i}] {a}'
                         for i, a in enumerate(answers)], fontsize=9)
    
    # Label memory regions
    ax4.axvline(x=topK - 0.5, color='#58a6ff', linestyle='--', alpha=0.7, linewidth=1.5)
    ax4.text(topK / 2, -0.7, '← Video Frames →', ha='center', fontsize=9, color='#58a6ff')
    ax4.text(topK + (mem_len - topK) / 2, -0.7, '← Question Tokens →',
             ha='center', fontsize=9, color='#3fb950')
    
    ax4.set_title('Step 3: Answer → Memory Cross-Attention (Why model prefers this answer)',
                  fontsize=12, fontweight='bold', color='#a371f7')
    plt.colorbar(im4, ax=ax4, label='Attention', shrink=0.8)
    
    # Highlight correct and predicted rows
    for idx, color in [(correct_idx, '#58a6ff'), (pred_idx, '#da3633')]:
        rect = plt.Rectangle((-0.5, idx - 0.5), mem_len, 1, linewidth=2,
                            edgecolor=color, facecolor='none', linestyle='--')
        ax4.add_patch(rect)
    
    path1 = f'{save_prefix}_dashboard.png'
    plt.savefig(path1, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'📊 Dashboard saved: {path1}')
    
    # ════════════════════════════════════════════════
    # FIGURE 2: Reasoning Explanation
    # ════════════════════════════════════════════════
    fig2, ax5 = plt.subplots(figsize=(16, 10))
    ax5.axis('off')
    
    # Build reasoning text
    top_frames = sorted(range(n_frames), key=lambda i: frame_imp[i], reverse=True)[:5]
    
    # Answer attention analysis
    ans_video_att = ans_att[:, :topK].mean(axis=1)  # avg attention to video frames per answer
    ans_text_att = ans_att[:, topK:].mean(axis=1)   # avg attention to question tokens per answer
    
    reasoning = []
    reasoning.append(f'{"═" * 70}')
    reasoning.append(f'  🔬 MODEL REASONING ANALYSIS')
    reasoning.append(f'{"═" * 70}')
    reasoning.append(f'')
    reasoning.append(f'  📹 Video: {video_id}')
    reasoning.append(f'  ❓ Question: {question}')
    reasoning.append(f'  📝 Type: {question_type}')
    reasoning.append(f'')
    reasoning.append(f'  ── Step 1: Frame Selection ──')
    reasoning.append(f'  Model selected frames: {list(sel_indices)}')
    reasoning.append(f'  Top-5 frames by attention: {top_frames}')
    reasoning.append(f'  → The model focuses on frames {list(sel_indices)} as most')
    reasoning.append(f'    relevant to the question.')
    reasoning.append(f'')
    reasoning.append(f'  ── Step 2: Object Analysis ──')
    for fi in range(min(topK, len(sel_indices))):
        top_objs = np.argsort(obj_imp[fi])[::-1][:3]
        scores = [f'obj{o}={obj_imp[fi, o]:.3f}' for o in top_objs]
        reasoning.append(f'  Frame {sel_indices[fi]}: Top objects → {", ".join(scores)}')
    reasoning.append(f'')
    reasoning.append(f'  ── Step 3: Answer Scoring ──')
    for i in range(5):
        marker = '✓' if i == correct_idx else ('✗◀' if i == pred_idx and i != correct_idx else ' ')
        reasoning.append(f'  [{i}] {marker} P={probs[i]:.4f} | vid_att={ans_video_att[i]:.4f}'
                        f' | txt_att={ans_text_att[i]:.4f} | {answers[i][:50]}')
    reasoning.append(f'')
    reasoning.append(f'  ── Conclusion ──')
    if pred_idx == correct_idx:
        reasoning.append(f'  ✅ Model correctly chose [{pred_idx}] with confidence {probs[pred_idx]:.3f}')
        reasoning.append(f'     The model attended strongly to video evidence supporting this answer.')
    else:
        reasoning.append(f'  ❌ Model chose [{pred_idx}] (P={probs[pred_idx]:.3f}) but correct is [{correct_idx}] (P={probs[correct_idx]:.3f})')
        diff = probs[pred_idx] - probs[correct_idx]
        reasoning.append(f'     Confidence gap: {diff:.4f}')
        if ans_video_att[pred_idx] > ans_video_att[correct_idx]:
            reasoning.append(f'     → Model over-attended to video evidence for wrong answer')
            reasoning.append(f'       (vid_att: wrong={ans_video_att[pred_idx]:.4f} vs correct={ans_video_att[correct_idx]:.4f})')
        else:
            reasoning.append(f'     → Model attended more to question tokens for wrong answer')
            reasoning.append(f'       (txt_att: wrong={ans_text_att[pred_idx]:.4f} vs correct={ans_text_att[correct_idx]:.4f})')
    reasoning.append(f'{"═" * 70}')
    
    text = '\n'.join(reasoning)
    ax5.text(0.02, 0.98, text, transform=ax5.transAxes, fontsize=10,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#0d1117',
                      edgecolor='#30363d', alpha=0.95),
             color='#c9d1d9')
    
    path2 = f'{save_prefix}_reasoning.png'
    plt.savefig(path2, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'🔬 Reasoning saved: {path2}')
    
    # Also print to console
    print('\n'.join(reasoning))
    
    return path1, path2

print('Visualization functions defined')

In [ ]:
# CELL 13: Single Sample Inference
# ============================================
# 🔴 SET YOUR VIDEO ID AND QUESTION TYPE HERE
# ============================================
INFERENCE_VIDEO_ID = None  # e.g., '9q8zoFUvTiw_000131_000141'
INFERENCE_QUESTION_TYPE = None  # e.g., 'descriptive', 'explanatory', 'predictive', etc.

# If None, automatically pick a wrong prediction from the CSV
if INFERENCE_VIDEO_ID is None:
    print('No video_id specified, picking an incorrect prediction from CSV...')
    incorrect_df_sorted = incorrect_df.sort_values('confidence', ascending=False)
    sample_row = incorrect_df_sorted.iloc[0]
    INFERENCE_VIDEO_ID = sample_row['video_id']
    INFERENCE_QUESTION_TYPE = sample_row['question_type']
    print(f'Selected: video={INFERENCE_VIDEO_ID}, type={INFERENCE_QUESTION_TYPE}')

print(f'\n=== Single Sample Inference ===')
print(f'Video ID: {INFERENCE_VIDEO_ID}')
print(f'Question Type: {INFERENCE_QUESTION_TYPE}')

# Load data for this video
vid = INFERENCE_VIDEO_ID
qtype = INFERENCE_QUESTION_TYPE

# Load frame features
ff_path = os.path.join(args.video_feature_root, f'{vid}.pt')
ff = torch.load(ff_path, weights_only=True).float()
nf = ff.shape[0]
if nf > args.topK_frame:
    indices = np.linspace(0, nf - 1, args.topK_frame).astype(int)
    ff = ff[indices]
elif nf < args.topK_frame:
    ff = torch.cat([ff, torch.zeros(args.topK_frame - nf, ff.shape[1])], 0)

# Load object features
from DataLoader import VideoQADataset
import pickle as pkl
gdino_path = os.path.join(args.grounding_dino_path, f'{vid}.pkl')
with open(gdino_path, 'rb') as f:
    gdata = pkl.load(f)
frames_data = gdata.get('frames', [])
orig_h = gdata.get('orig_h', 1080)
orig_w = gdata.get('orig_w', 1920)
gnf = len(frames_data)
if gnf > args.topK_frame:
    gindices = np.linspace(0, gnf - 1, args.topK_frame).astype(int)
else:
    gindices = range(gnf)

objs = []
for idx in gindices:
    fd = frames_data[idx]
    roi = fd.get('roi_features', np.zeros((0, 1024), dtype=np.float32))
    boxes = fd.get('boxes_xyxy_orig', np.zeros((0, 4), dtype=np.float32))
    if len(boxes) > 0:
        boxes_n = boxes / np.array([orig_w, orig_h, orig_w, orig_h], dtype=np.float32)
    else:
        boxes_n = boxes
    if len(roi) > 0:
        of_feat = np.concatenate([roi, boxes_n], axis=-1)
    else:
        of_feat = np.zeros((0, 1028), dtype=np.float32)
    of_feat = torch.from_numpy(of_feat).float()
    N = of_feat.shape[0]
    if N > args.objs:
        of_feat = of_feat[:args.objs]
    elif N < args.objs:
        of_feat = torch.cat([of_feat, torch.zeros(args.objs - N, 1028)], 0)
    objs.append(of_feat)
while len(objs) < args.topK_frame:
    objs.append(torch.zeros(args.objs, 1028))
of = torch.stack(objs)

# Load annotation
from DataLoader import FAMILY_TO_ID
ann_path = os.path.join(args.sample_list_path, vid)
with open(os.path.join(ann_path, 'text.json'), encoding='utf-8') as f:
    td = json.load(f)
with open(os.path.join(ann_path, 'answer.json'), encoding='utf-8') as f:
    ad = json.load(f)

# Determine the base type (strip '_reason' suffix)
base_type = qtype.replace('_reason', '')
q_data = td[base_type]
a_data = ad[base_type]

question = q_data['question']
if 'reason' in qtype:
    answer_choices = q_data['reason']
    correct_idx = a_data['reason']
else:
    answer_choices = q_data['answer']
    correct_idx = a_data['answer']

q_family_id = FAMILY_TO_ID.get(qtype, 0)

# Build input format
qns_word = (question,)
ans_word = [tuple(f"{question} [SEP] {c}" for c in answer_choices)]

print(f'\nQuestion: {question}')
print(f'Answers:')
for i, a in enumerate(answer_choices):
    marker = '✓' if i == correct_idx else ' '
    print(f'  [{i}] {marker} {a}')

# Run inference
ff_batch = ff.unsqueeze(0).to(device)
of_batch = of.unsqueeze(0).to(device)

result = inference_with_attention(
    model, ff_batch, of_batch, qns_word, ans_word, device,
    q_family_id=q_family_id
)

print(f'\n═══ Inference Result ═══')
print(f'Predicted: [{result["pred"]}] {answer_choices[result["pred"]]}')
print(f'Correct:   [{correct_idx}] {answer_choices[correct_idx]}')
print(f'Probabilities: {[f"{p:.4f}" for p in result["probs"]]}')
print(f'Selected frames: {list(result["selected_frame_indices"])}')

In [ ]:
# CELL 14: Visualize Attention
print('=== Generating Attention Visualizations ===')

save_prefix = f'attention_{vid}_{qtype}'
visualize_full_attention(
    result=result,
    video_id=vid,
    question=question,
    answers=answer_choices,
    correct_idx=correct_idx,
    question_type=qtype,
    save_prefix=save_prefix
)

In [ ]:
# CELL 15: Analyze Multiple Wrong Predictions
print('=== Batch Analysis of Wrong Predictions ===')

# Pick top-5 most confident wrong predictions (model is very sure but wrong)
NUM_ANALYZE = 3
confident_wrong = incorrect_df.sort_values('confidence', ascending=False).head(NUM_ANALYZE)

for row_idx, (_, row) in enumerate(confident_wrong.iterrows()):
    vid_i = row['video_id']
    qtype_i = row['question_type']
    print(f'\n{"═" * 60}')
    print(f'  Wrong Prediction #{row_idx + 1}: {vid_i} ({qtype_i})')
    print(f'  Confidence: {row["confidence"]:.4f}')
    print(f'{"═" * 60}')
    
    try:
        # Load data
        ff_i = torch.load(os.path.join(args.video_feature_root, f'{vid_i}.pt'), weights_only=True).float()
        nf_i = ff_i.shape[0]
        if nf_i > args.topK_frame:
            ff_i = ff_i[np.linspace(0, nf_i - 1, args.topK_frame).astype(int)]
        elif nf_i < args.topK_frame:
            ff_i = torch.cat([ff_i, torch.zeros(args.topK_frame - nf_i, ff_i.shape[1])], 0)
        
        gdino_i = os.path.join(args.grounding_dino_path, f'{vid_i}.pkl')
        with open(gdino_i, 'rb') as f:
            gd_i = pkl.load(f)
        fd_i = gd_i.get('frames', [])
        oh_i, ow_i = gd_i.get('orig_h', 1080), gd_i.get('orig_w', 1920)
        gnf_i = len(fd_i)
        gi = np.linspace(0, gnf_i - 1, args.topK_frame).astype(int) if gnf_i > args.topK_frame else range(gnf_i)
        
        objs_i = []
        for idx in gi:
            roi_i = fd_i[idx].get('roi_features', np.zeros((0, 1024), dtype=np.float32))
            box_i = fd_i[idx].get('boxes_xyxy_orig', np.zeros((0, 4), dtype=np.float32))
            bn_i = box_i / np.array([ow_i, oh_i, ow_i, oh_i], dtype=np.float32) if len(box_i) > 0 else box_i
            of_i = np.concatenate([roi_i, bn_i], axis=-1) if len(roi_i) > 0 else np.zeros((0, 1028), dtype=np.float32)
            of_i = torch.from_numpy(of_i).float()
            if of_i.shape[0] > args.objs:
                of_i = of_i[:args.objs]
            elif of_i.shape[0] < args.objs:
                of_i = torch.cat([of_i, torch.zeros(args.objs - of_i.shape[0], 1028)], 0)
            objs_i.append(of_i)
        while len(objs_i) < args.topK_frame:
            objs_i.append(torch.zeros(args.objs, 1028))
        of_stk = torch.stack(objs_i)
        
        # Load annotation
        bt_i = qtype_i.replace('_reason', '')
        with open(os.path.join(args.sample_list_path, vid_i, 'text.json'), encoding='utf-8') as f:
            td_i = json.load(f)
        with open(os.path.join(args.sample_list_path, vid_i, 'answer.json'), encoding='utf-8') as f:
            ad_i = json.load(f)
        
        q_i = td_i[bt_i]['question']
        if 'reason' in qtype_i:
            ac_i = td_i[bt_i]['reason']
            ci_i = ad_i[bt_i]['reason']
        else:
            ac_i = td_i[bt_i]['answer']
            ci_i = ad_i[bt_i]['answer']
        
        qfi_i = FAMILY_TO_ID.get(qtype_i, 0)
        aw_i = [tuple(f"{q_i} [SEP] {c}" for c in ac_i)]
        
        res_i = inference_with_attention(
            model, ff_i.unsqueeze(0).to(device), of_stk.unsqueeze(0).to(device),
            (q_i,), aw_i, device, q_family_id=qfi_i
        )
        
        visualize_full_attention(
            result=res_i, video_id=vid_i, question=q_i,
            answers=ac_i, correct_idx=ci_i, question_type=qtype_i,
            save_prefix=f'wrong_{row_idx}_{vid_i}_{qtype_i}'
        )
    except Exception as e:
        print(f'  ⚠️ Error analyzing {vid_i}: {e}')
        import traceback
        traceback.print_exc()

print('Batch analysis complete!')